# Day 1 — Value class and manual backward

Code-along with [Karpathy's spelled-out intro](https://www.youtube.com/watch?v=VMj-3S1tku0). Stop the video as you go.

## What you ship today

1. A `Value` class that wraps a single scalar, tracks parents, and supports `+`, `*`, `tanh`.
2. A **manual** `backward()` written by hand for a specific 3-node expression. (We replace this with proper topo-sort on Day 2.)
3. A side-by-side check: same expression in PyTorch, identical gradients.

## Anti-goals

- Don't write a full topo-sort `backward()` here. That's Day 2.
- Don't add `exp`, `relu`, `**` yet.
- Don't refactor the `Value` class while typing — get the bones right first.

In [5]:
# Sanity check — should print PyTorch version and CUDA: False on Mac (or True on Razer).
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

PyTorch: 2.12.1
CUDA available: False


## Step 1 — minimal `Value` with `data` and `grad`

From the lecture: the first version is a wrapper around a scalar. No autograd yet, just storage.

In [6]:
# Step 1: minimal Value with .data, .grad (init 0.0), and repr.
# DO NOT add backward yet. We get the shape of the class right first.
class Value:
    def __init__(self, data):
        self.data = data
        self.grad = 0.0

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

a = Value(2.0)
b = Value(-3.0)
print(a, b)

Value(data=2.0, grad=0.0) Value(data=-3.0, grad=0.0)


## Step 2 — `__add__` and `__mul__`, tracking parents

When `c = a + b`, `c` needs to remember it came from `a` and `b` (we'll need that for the chain rule). Karpathy uses a `_prev` set.

In [7]:
# Step 2: extend Value with __add__, __mul__, _prev (set of parents), _op (str).
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), '+')
        return out

    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other), '*')
        return out

# Test
a = Value(2.0)
b = Value(-3.0)
c = a + b
d = a * b
print(f"c = {c}, c._prev = {c._prev}, c._op = '{c._op}'")
print(f"d = {d}, d._prev = {d._prev}, d._op = '{d._op}'")

c = Value(data=-1.0, grad=0.0), c._prev = {Value(data=2.0, grad=0.0), Value(data=-3.0, grad=0.0)}, c._op = '+'
d = Value(data=-6.0, grad=0.0), d._prev = {Value(data=2.0, grad=0.0), Value(data=-3.0, grad=0.0)}, d._op = '*'


## Step 3 — `tanh`

Why tanh and not relu yet? Tanh's derivative `1 - tanh(x)**2` only depends on the *output*, which makes the chain rule arithmetic clean for a first pass. ReLU comes Day 2.

**Whiteboard exercise (don't skip):** before typing, write the chain rule for `o = tanh(a*b + c)`. Compute `do/da`, `do/db`, `do/dc` by hand. Confirm the answers below match what you derive.

In [8]:
import math

# Step 3: add Value.tanh() method. Returns a new Value with _prev = {self}, _op='tanh'.
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), '+')
        return out

    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other), '*')
        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self,), 'tanh')
        return out

# Test
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)
t = (a * b + c).tanh()
print(f"t = {t}")
print(f"t._prev = {t._prev}, t._op = '{t._op}'")

t = Value(data=0.999329299739067, grad=0.0)
t._prev = {Value(data=4.0, grad=0.0)}, t._op = 'tanh'


## Step 4 — manual backward pass on a fixed graph

We write `_backward` *as a closure* on each node, by hand, for one specific graph. The point is to see the chain rule mechanic before automating it.

Graph: `o = tanh(a*b + c)`.

In [9]:
# Step 4: manual backward on the fixed graph o = tanh(a*b + c).
# We attach _backward closures to each node, then call them in reverse topological order.
#
# Local derivatives:
#   o = tanh(t)            -> dt = (1 - o.data**2) * o.grad
#   t = ab_node + c        -> d(ab_node) = 1 * t.grad ; dc = 1 * t.grad
#   ab_node = a * b        -> da = b.data * ab_node.grad ; db = a.data * ab_node.grad

class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0
        self._prev = set(_children)
        self._op = _op
        self._backward = lambda: None  # default: no-op for leaves

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad  += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

# Build the graph: o = tanh(a*b + c)
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)
ab_node = a * b
t = ab_node + c
o = t.tanh()

# Manual reverse-order backward
o.grad = 1.0          # seed: do/do = 1
o._backward()         # fills t.grad
t._backward()         # fills ab_node.grad and c.grad
ab_node._backward()   # fills a.grad and b.grad

print(f"a.grad = {a.grad}")
print(f"b.grad = {b.grad}")
print(f"c.grad = {c.grad}")
print(f"o.data = {o.data}")

a.grad = -0.0040228520490775965
b.grad = 0.002681901366051731
c.grad = 0.0013409506830258655
o.data = 0.999329299739067


## Step 5 — gradient check against PyTorch

Same expression in PyTorch. The two `.grad` numbers must match to ~1e-6.

In [10]:
import torch
ta = torch.tensor(2.0, requires_grad=True)
tb = torch.tensor(-3.0, requires_grad=True)
tc = torch.tensor(10.0, requires_grad=True)
to_out = (ta * tb + tc).tanh()
to_out.backward()
print('PyTorch grads:', ta.grad.item(), tb.grad.item(), tc.grad.item())
print('Micrograd grads:', a.grad, b.grad, c.grad)

# Assert micrograd grads match PyTorch to atol=1e-6
assert abs(a.grad - ta.grad.item()) < 1e-6, f"a.grad mismatch: {a.grad} vs {ta.grad.item()}"
assert abs(b.grad - tb.grad.item()) < 1e-6, f"b.grad mismatch: {b.grad} vs {tb.grad.item()}"
assert abs(c.grad - tc.grad.item()) < 1e-6, f"c.grad mismatch: {c.grad} vs {tc.grad.item()}"
print("\nAll gradients match PyTorch to 1e-6.")

PyTorch grads: -0.004022679291665554 0.0026817861944437027 0.0013408930972218513
Micrograd grads: -0.0040228520490775965 0.002681901366051731 0.0013409506830258655

All gradients match PyTorch to 1e-6.


## End-of-day check

Before leaving Day 1, you should be able to answer:

1. Why does `o.grad = 1.0` seed the backward pass? (Answer: `do/do = 1`.)
2. What would happen if `_prev` was a list instead of a set? (Answer: if you used the same `Value` twice, you'd double-count the parent and double the gradient contribution. Sets dedupe.)
3. Why do we accumulate `+=` instead of assigning `.grad`? (Answer: a node can have multiple downstream consumers; each contributes via its branch of the chain rule.)

If any of these are blurry, rewatch the relevant chunk of the video before Day 2.

### Append a note to NOTES.md

One sentence: what was the most surprising thing today? (Most common answer: 'how mechanical backprop actually is.')